# M4 — conditional RBF-kernel copula

This notebook explains the fifth principal dependence family. M4 represents dependence as geometric similarity in a learned entity space. Its fitting partitions and evaluation status are the same as those of M2 and M3. See [Chapter 4](../../docs/scientific/04_dependence_models.md).


## Notation and kernel hypothesis

For forecast instance $i$, lead $\tau$, and static group $\mathcal E_g$, a shared network maps each information vector to an embedding
$$h_{k,\tau}^{(i)}=g_\theta(v_{k,\tau}^{(i)})\in\mathbb R^d.$$
For entities $a$ and $b$, M4 defines the kernel covariance
$$K_{ab}=\exp[-\frac{\lVert h_a-h_b\rVert_2^2}{2\ell^2}]+\nu\,\mathbf1(a=b),$$
then normalizes $K$ to $R_{g,\tau}^{(i)}\in\mathbb R^{K_g\times K_g}$. Here $\ell>0$ is the learned length scale and $\nu\ge0$ is the nugget. The RBF Gram matrix is positive semidefinite and permutation equivariant. Its off-diagonal elements are nonnegative, so this family cannot represent negative residual rank association.


## Meaning of the length scale

At embedded distance $d_{ab}=\lVert h_a-h_b\rVert_2$, the off-diagonal kernel similarity is $\exp[-d_{ab}^2/(2\ell^2)]$. Small $\ell$ allows correlation to decay rapidly; large $\ell$ preserves similarity over greater learned distances. This distance is learned from FM-derived features and should not be interpreted automatically as geographical distance.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').is_file()
)
CACHE_ROOT = PROJECT_ROOT / 'artifacts' / 'cache'
SOURCE_ROOT = str(PROJECT_ROOT / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

from simcast.config import load_base_config, load_method_config, resolve_run_config
from simcast.experiments import locate_compatible_cache
from simcast.fm.cache import load_pit_library
from simcast.fm.pit import dependence_pit_scores
from simcast.sampling.gaussian_copula import generate_scenarios, sample_gaussian_uniforms
from simcast.training.checkpoint import load_conditional_checkpoint
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config

BASE_FILE = 'configs/bases/liander2024/transformer.yaml'
METHOD_FILE = 'configs/methods/m4_conditional_kernel.yaml'
BASE_OVERRIDES = (f'output.cache_dir={CACHE_ROOT}',)
METHOD_OVERRIDES = ()
FIT_IF_MISSING = False
RUN_EVALUATION = False
VISUALIZE_FITTED_MODEL = True
RUN_DIR = PROJECT_ROOT / 'runs' / 'notebook_walkthrough' / 'm4_conditional_kernel'

base = load_base_config(PROJECT_ROOT / BASE_FILE, overrides=BASE_OVERRIDES)
method = load_method_config(PROJECT_ROOT / METHOD_FILE, overrides=METHOD_OVERRIDES)
config = resolve_run_config(base, method)
cache_dir = locate_compatible_cache(base)
library = load_pit_library(cache_dir, access='training')
ds = library.dataset
entity_ids = [str(value) for value in ds.entity_id.values]
K_g = len(entity_ids)
s = method.model
print({'g': base.data.entity_type, 'K_g': K_g,
       'embedding dimension': s.embedding_dim, 'epochs': method.optimization.epochs})


## A one-dimensional kernel example

The curves below isolate the role of $\ell$ before any network is fitted. Each begins at similarity one when two embeddings coincide and decays toward zero as their distance increases. The example describes the kernel family, not an estimated result.


In [ ]:
distance = np.linspace(0, 4, 400)
fig, ax = plt.subplots(figsize=(8, 4))
for length_scale in (0.5, 1.0, 2.0):
    similarity = np.exp(-distance**2 / (2 * length_scale**2))
    ax.plot(distance, similarity, label=fr'$\ell={length_scale:g}$')
ax.set(xlabel='learned embedding distance $d_{ab}$', ylabel='RBF similarity',
       title='Kernel similarity under alternative length scales')
ax.legend()
plt.show()


## Estimation

M4 uses the same Gaussian-copula pseudo-likelihood as M2 and M3:
$$\mathcal L=\tfrac12[\log|R|+\mathbf z^\mathsf T(R^{-1}-I)\mathbf z].$$
It is fitted on every valid complete training vector and selected using chronological validation pseudo-negative log-likelihood. There is no M4-specific origin truncation or implicit optimization cap. The following cell calls the same functions as the command-line workflow and remains inert unless enabled.


In [ ]:
if FIT_IF_MISSING and not (RUN_DIR / 'best.pt').is_file():
    train_from_config(config, cache_dir=cache_dir, output_dir=RUN_DIR)
if RUN_EVALUATION:
    evaluate_from_config(
        config,
        methods=('conditional_kernel',),
        method_runs={'conditional_kernel': RUN_DIR},
        cache_dir=cache_dir,
        output_dir=RUN_DIR / 'evaluation',
    )


## Conditional correlation across information states

A fitted M4 checkpoint maps each complete information state to learned entity embeddings and therefore to $R_{g,\tau}^{(i)}$. The first two panels compare two cached states. The third shows the signed difference between their conditional correlation matrices.

The computation requires `RUN_DIR/best.pt`. If it is absent, point `RUN_DIR` to a compatible fitted M4 run or explicitly enable `FIT_IF_MISSING`. An untrained network is never presented as a scientific result.


In [ ]:
checkpoint_path = RUN_DIR / 'best.pt'
m4_visuals_ready = VISUALIZE_FITTED_MODEL and checkpoint_path.is_file()

if m4_visuals_ready:
    checkpoint = load_conditional_checkpoint(checkpoint_path)
    if checkpoint.method != 'conditional_kernel':
        raise ValueError('The selected checkpoint is not an M4 model.')
    if checkpoint.entity_ids != tuple(entity_ids):
        raise ValueError('The checkpoint entity order does not match the cache.')

    training = np.asarray(ds['split'].values) == 'train'
    valid = training[:, None] & np.asarray(ds['pit_valid'].values, dtype=bool)
    candidates = np.argwhere(valid)
    if len(candidates) == 0:
        raise ValueError('No complete training case is available for visualization.')

    positions = np.linspace(0, len(candidates) - 1, min(2, len(candidates)), dtype=int)
    selected = candidates[np.unique(positions)]
    levels = torch.tensor(np.asarray(ds['quantile'].values).copy(), dtype=torch.float32)
    locations = None
    if 'latitude' in ds and 'longitude' in ds:
        coordinates = np.stack((ds['latitude'].values, ds['longitude'].values), axis=-1)
        locations = torch.tensor(coordinates.copy(), dtype=torch.float32)

    matrices = []
    learned_embeddings = []
    case_features = []
    with torch.no_grad():
        for origin_index, lead_index in selected:
            embeddings = torch.tensor(
                np.asarray(ds['forecast_embedding'].values[origin_index:origin_index + 1]).copy(),
                dtype=torch.float32,
            )
            predictions = torch.tensor(
                np.asarray(ds['quantile_prediction'].values[origin_index:origin_index + 1]).copy(),
                dtype=torch.float32,
            )
            features = checkpoint.feature_builder.transform(
                embeddings, predictions, levels, locations
            )[0, :, lead_index]
            case_features.append(features)
            learned_embeddings.append(checkpoint.model.entity_embeddings(features).numpy())
            matrices.append(checkpoint.model(features).numpy())

    if len(matrices) == 1:
        matrices.append(matrices[0].copy())
        learned_embeddings.append(learned_embeddings[0].copy())
        selected = np.vstack((selected, selected))

    difference = matrices[1] - matrices[0]
    difference_limit = max(float(np.abs(difference).max()), 1e-6)
    labels = [
        f'origin {ds.origin_timestamp.values[i]}, lead {int(ds.lead.values[tau])}'
        for i, tau in selected
    ]
    length_scale = float(checkpoint.model.length_scale.detach())

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, matrix, label in zip(axes[:2], matrices, labels):
        image = ax.imshow(matrix, vmin=-1, vmax=1, cmap='coolwarm')
        ax.set(title=label, xlabel='entity index', ylabel='entity index')
    difference_image = axes[2].imshow(
        difference, vmin=-difference_limit, vmax=difference_limit, cmap='coolwarm'
    )
    axes[2].set(title=r'$R^{(i_b)}-R^{(i_a)}$',
                xlabel='entity index', ylabel='entity index')
    fig.colorbar(image, ax=axes[:2], label='correlation', shrink=0.8)
    fig.colorbar(difference_image, ax=axes[2], label='correlation change', shrink=0.8)
    fig.suptitle(fr'M4 dependence across states: learned $\ell={length_scale:.3g}$')
    plt.show()
else:
    print(f'No fitted M4 checkpoint at {checkpoint_path}.')


## Learned embedding geometry and kernel restriction

For the first selected state, the left panel gives a two-dimensional principal projection of the learned embeddings $h_k$. Nearby points have similar learned representations; the displayed axes are only visualization coordinates and have no direct physical units. The right panel verifies the kernel mechanism by plotting every entity-pair distance against its resulting correlation. All pairs must lie on the fitted RBF curve after nugget normalization.


In [ ]:
if m4_visuals_ready:
    embedded = learned_embeddings[0]
    centered = embedded - embedded.mean(axis=0, keepdims=True)
    _, _, directions = np.linalg.svd(centered, full_matrices=False)
    coordinates = centered @ directions[:2].T
    if coordinates.shape[1] == 1:
        coordinates = np.column_stack((coordinates[:, 0], np.zeros(K_g)))

    distances = np.linalg.norm(embedded[:, None] - embedded[None, :], axis=-1)
    upper = np.triu_indices(K_g, 1)
    scale = 1 + checkpoint.model.nugget + checkpoint.model.jitter
    distance_grid = np.linspace(0, max(float(distances[upper].max()), 1e-6), 300)
    kernel_curve = np.exp(-distance_grid**2 / (2 * length_scale**2)) / scale

    fig, (ax_space, ax_kernel) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax_space.scatter(coordinates[:, 0], coordinates[:, 1], s=45)
    for index, entity_id in enumerate(entity_ids):
        ax_space.annotate(entity_id, coordinates[index], xytext=(4, 4),
                          textcoords='offset points', fontsize=8)
    ax_space.set(xlabel='principal embedding coordinate 1',
                 ylabel='principal embedding coordinate 2',
                 title='Learned entity geometry')

    ax_kernel.scatter(distances[upper], matrices[0][upper], label='entity pairs')
    ax_kernel.plot(distance_grid, kernel_curve, color='C1', label='fitted RBF law')
    ax_kernel.set(xlabel=r'learned distance $\Vert h_a-h_b\Vert_2$',
                  ylabel=r'correlation $R_{ab}$',
                  title='Distance determines nonnegative correlation')
    ax_kernel.legend()
    fig.suptitle('M4 embedding and kernel mechanism for one information state')
    plt.tight_layout()
    plt.show()
else:
    print('A fitted M4 checkpoint is required for the embedding geometry.')


## Historical scores and one conditional Gaussian law

The left panel shows complete historical training scores at the selected lead, pooled across origins. The right panel shows draws from the M4 Gaussian copula for the first selected state. The displayed pair has the largest off-diagonal correlation. Because one panel mixes information states and the other is conditional, the comparison explains the fitted mechanism rather than testing out-of-sample calibration.


In [ ]:
if m4_visuals_ready:
    origin_index, lead_index = map(int, selected[0])
    R_fitted = matrices[0]
    strength = R_fitted.copy()
    np.fill_diagonal(strength, -np.inf)
    entity_a, entity_b = np.unravel_index(np.argmax(strength), strength.shape)

    scores, _ = dependence_pit_scores(
        torch.tensor(np.asarray(ds['pit_u'].values).copy()),
        torch.tensor(np.asarray(ds['pit_z'].values).copy()),
        torch.tensor(training.copy()), levels,
        mode=config.pit.dependence_transform, eps=config.pit.eps,
    )
    observed = scores.numpy()[training, :, lead_index]
    observed = observed[np.isfinite(observed).all(axis=1)]

    generator = torch.Generator().manual_seed(config.sampling.evaluation_seed)
    fitted_u = sample_gaussian_uniforms(
        torch.tensor(R_fitted, dtype=torch.float32), len(observed), generator=generator
    )
    fitted_z = torch.special.ndtri(fitted_u).numpy()
    step = max(1, len(observed) // 5000)
    observed_pair = observed[::step][:, [entity_a, entity_b]]
    fitted_pair = fitted_z[::step][:, [entity_a, entity_b]]
    limit = max(3.5, np.nanpercentile(
        np.abs(np.vstack((observed_pair, fitted_pair))), 99.5
    ))

    observed_r = np.corrcoef(observed, rowvar=False)[entity_a, entity_b]
    fitted_r = np.corrcoef(fitted_z, rowvar=False)[entity_a, entity_b]
    panels = (
        (observed_pair, fr'Observed training mixture: $r={observed_r:.2f}$'),
        (fitted_pair, fr'M4 selected context: $r={fitted_r:.2f}$'),
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
    for ax, (points, title) in zip(axes, panels):
        ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.2)
        ax.axhline(0, color='0.75', lw=0.8)
        ax.axvline(0, color='0.75', lw=0.8)
        ax.set(xlim=(-limit, limit), ylim=(-limit, limit), aspect='equal',
               xlabel=fr'$z_{{{entity_a + 1}}}$', title=title)
    axes[0].set_ylabel(fr'$z_{{{entity_b + 1}}}$')
    fig.suptitle(
        f'M4 score dependence: {entity_ids[entity_a]} versus {entity_ids[entity_b]}',
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()
else:
    print('A fitted M4 checkpoint is required for the score-space comparison.')


## From conditional ranks to entity and aggregate scenarios

For the same state, M4 draws $X^{(m)}\sim\mathcal N(\mathbf0,R_{g,\tau}^{(i)})$, transforms it to $U_k^{(m)}=\Phi(X_k^{(m)})$, and projects each rank through the fixed marginal forecast. This yields entity scenarios $\widetilde Y_{k,\tau}^{(i,m)}$ and aggregate scenarios $\widetilde A_{g,\tau}^{(i,m)}=\sum_k\widetilde Y_{k,\tau}^{(i,m)}$. The three panels follow the same draws through probability, entity, and aggregate space.


In [ ]:
if m4_visuals_ready:
    qhat = torch.tensor(
        np.asarray(ds['quantile_prediction'].values[origin_index, :, lead_index]).copy(),
        dtype=torch.float32,
    )
    generator = torch.Generator().manual_seed(config.sampling.evaluation_seed)
    scenarios = generate_scenarios(
        torch.tensor(R_fitted, dtype=torch.float32), qhat, levels,
        num_samples=config.sampling.num_samples,
        generator=generator, marginal_mode=config.pit.mode,
    )
    u = scenarios.uniforms.numpy()
    y = scenarios.entity_samples.numpy()
    aggregate = scenarios.aggregate_samples.numpy()
    observed = float(ds['true_y'].values[origin_index, :, lead_index].sum())
    q05, median, q95 = np.quantile(aggregate, [0.05, 0.5, 0.95])
    lead = int(ds['lead'].values[lead_index])
    origin = ds['origin_timestamp'].values[origin_index]

    fig, (ax_u, ax_y, ax_a) = plt.subplots(1, 3, figsize=(15, 4.5))
    ax_u.scatter(u[:, entity_a], u[:, entity_b], s=7, alpha=0.2)
    ax_u.set(xlim=(0, 1), ylim=(0, 1), aspect='equal',
             xlabel=fr'$U_{{{entity_a + 1}}}$', ylabel=fr'$U_{{{entity_b + 1}}}$',
             title='M4 conditional copula ranks')
    ax_y.scatter(y[:, entity_a], y[:, entity_b], s=7, alpha=0.2)
    ax_y.set(xlabel=entity_ids[entity_a], ylabel=entity_ids[entity_b],
             title='Projected entity scenarios')
    ax_a.hist(aggregate, bins=35, density=True, alpha=0.75)
    ax_a.axvline(observed, color='C3', ls=':', label=fr'observed $a={observed:.3g}$')
    ax_a.axvline(median, color='black', ls='--', label=fr'median $={median:.3g}$')
    ax_a.axvspan(q05, q95, color='C0', alpha=0.15, label='central 90% interval')
    ax_a.set(xlabel=fr'$\widetilde A_{{g,{lead}}}^{{(i,m)}}$',
             ylabel='empirical density', title='Induced aggregate distribution')
    ax_a.legend(fontsize=8)
    fig.suptitle(f'M4 scenario path: origin {origin}, lead {lead}', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('A fitted M4 checkpoint is required for scenario propagation.')


## Controls and interpretation

`method.features.*` defines $v_k$. `method.model.embedding_dim`, `hidden_dims`, `dropout`, `initial_length_scale`, `nugget`, and `jitter` define the learned kernel. `method.optimization.*` controls estimation. `ConditionalKernelGaussianCopula` implements M4. A poor result may indicate that nonnegative RBF similarity is too restrictive for the observed conditional association; this is a scientific limitation of the family, not a reduced execution status.
